# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` values as per best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()  # to_json returns a dict for display
print(f"{metadata['name']}\n\nDescription: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. For each record set, list its `@id`, available fields, and columns (all by `@id`).

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    Field @id: {f['@id']} | Name: {f.get('name', f['@id'])}")
    if 'columns' in rs:
        print("  Columns:")
        for c in rs['columns']:
            print(f"    Column @id: {c['@id']} | Name: {c.get('name', c['@id'])}")
    print("")

## 3. Data Extraction
Load data from specific record sets using `mlcroissant`. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all available record sets
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for Record Set @id: {rs_id}, Shape: {dataframes[rs_id].shape}")
            print(f"Columns (@id): {dataframes[rs_id].columns.tolist()}")
            print(dataframes[rs_id].head(), '\n')
        else:
            print(f"No records found for Record Set @id: {rs_id}\n")
    except Exception as e:
        print(f"Error loading records for Record Set @id: {rs_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping, using field and column `@id` identifiers.

In [ ]:
# Identify a DataFrame and numeric field for demonstration
# Let's assume there's a record set with regression results; select by @id
example_rs_id = None
numeric_field_id = None
group_field_id = None

# Search candidate DataFrame for a numeric field
for rsid, df in dataframes.items():
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in ['i', 'f']]
    if numeric_candidates:
        example_rs_id = rsid
        numeric_field_id = numeric_candidates[0]  # pick first numeric field
        # Try to find a categorical field
        category_candidates = [col for col in df.columns if df[col].dtype == 'object']
        if category_candidates:
            group_field_id = category_candidates[0]
        break

if example_rs_id and numeric_field_id:
    print(f"Using Record Set @id: {example_rs_id}, Numeric Field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = dataframes[example_rs_id][dataframes[example_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head(), '\n')

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(), '\n')

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric or categorical field found for EDA.")

## 5. Visualization
Visualize numeric data distributions and relationships between fields, with axes labeled by the fields' `@id` for clarity.

In [ ]:
# Visualize the selected numeric field
if example_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    dataframes[example_rs_id][numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped data available, bar plot
    if group_field_id and group_field_id in dataframes[example_rs_id].columns:
        grouped_mean = dataframes[example_rs_id].groupby(group_field_id)[numeric_field_id].mean()
        grouped_mean.plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable fields for visualization.")

## 6. Conclusion
This exploration demonstrated how to load, examine, and process the FAIR^2 dataset with `mlcroissant`, referencing record sets and fields exclusively by `@id`. We reviewed available entities, extracted records, performed EDA (including normalization and grouping), and visualized distributions.

**Dataset Limitations:**
- Some fields contain missing values and potential survey biases, as described in the metadata.
- Modeling results reflect only surveyed wards and capture limited explained variance.

For more advanced analysis or modeling, consult the dataset schema for further record set and field definitions by `@id`.